# SGAM Injection and Structural Attention Bias

This notebook implements the SGAM stage described in Phase 2 of the HiSE plan. The goal is to add a structural attention bias before softmax so structurally related tokens are boosted and structurally distant tokens are penalized while keeping the base encoder untouched.

Key design choices:
- weights are learned scalars with negative values at initialization to penalize distance
- SGAM is applied as ``softmax((QK^T / sqrt(d_k)) + SGAM)``
- shared weights are the default; per-layer weights are an extension for later ablation
- world-graph priors can warm-start the learned weights


In [ ]:
# Section 0: setup
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
for candidate in [repo_root, repo_root.parent, repo_root / "hise", repo_root.parent / "hise"]:
    if candidate.exists():
        src = candidate / "src" if candidate.name != "src" else candidate
        if src.exists():
            sys.path.insert(0, str(src))
        hise_src = candidate / "hise" / "src" if (candidate / "hise").exists() else None
        if hise_src is not None and hise_src.exists():
            sys.path.insert(0, str(hise_src))

import torch
import numpy as np

print("repo_root =", repo_root)
print("torch =", torch.__version__)


## 1. SGAM core formulation

For each pair of tokens i and j, compute the structural bias as:

``SGAM(i, j) = w_depth * |d_i - d_j| + w_parent * 1[parent_i != parent_j] + w_sibling * |s_i - s_j|``

The bias is negative for structurally different items, so it penalizes mismatched attention before the softmax. The default optimizer initialization keeps the weights negative, matching the intended effect.


In [ ]:
# Section 1: SGAM bias computation
import torch
from hise.address import StructuralAddress
from hise.injection import SGAMBias, sgam_bias

addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
    StructuralAddress(3, 2, 21, 1),
]

# Negative weights intentionally penalize structural distance before softmax.
bias = sgam_bias(addresses, depth_weight=-1.0, parent_weight=-2.0, sibling_weight=-0.5)
print("bias matrix:")
print(bias)
print("symmetric =", bool(torch.allclose(bias, bias.T)))


## 2. World-graph warm start for SGAM

The Phase 1 world graph can provide a prior over structural relations. If the graph says two concepts are siblings, then we want a stronger magnitude for ``w_sibling``. If two nodes are at different depths, then ``w_depth`` should also be stronger in magnitude.

This is an optional warm-start, not a hard constraint. It is useful for training stability and fast convergence on the earliest batches.


In [ ]:
# Section 2: optional world-graph warm start
from hise.injection import SGAMBias

prior = {"depth": -1.5, "parent": -2.0, "sibling": -1.2}
module = SGAMBias(depth_weight=-0.1, parent_weight=-0.1, sibling_weight=-0.1)
module.warm_start(depth=prior["depth"], parent=prior["parent"], sibling=prior["sibling"])
print("warm-start weights =", module.depth_weight.item(), module.parent_weight.item(), module.sibling_weight.item())


In [ ]:
# Section 3: BERT-style attention integration
import torch
from hise.address import StructuralAddress
from hise.injection import SGAMBias

class ToyBertSelfAttention(torch.nn.Module):
    def __init__(self, *, shared=True):
        super().__init__()
        self.shared = shared
        if shared:
            self.sgam = SGAMBias(depth_weight=-1.0, parent_weight=-2.0, sibling_weight=-0.5)
        else:
            self.sgam = [SGAMBias(depth_weight=-1.0, parent_weight=-2.0, sibling_weight=-0.5) for _ in range(12)]

    def forward(self, q, k, v, addresses):
        scores = (q @ k.transpose(-2, -1)) / q.shape[-1] ** 0.5
        bias = self.sgam(addresses) if self.shared else self.sgam[0](addresses)
        attn = torch.softmax(scores + bias, dim=-1)
        return attn @ v

addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
]

q = torch.randn(3, 4)
k = torch.randn(3, 4)
v = torch.randn(3, 4)
layer = ToyBertSelfAttention(shared=True)
out = layer(q, k, v, addresses)
print("attention output shape =", out.shape)
print(out)


In [ ]:
# Section 4: gradient verification
import torch
from hise.address import StructuralAddress
from hise.injection import SGAMBias

module = SGAMBias(depth_weight=-1.0, parent_weight=-2.0, sibling_weight=-0.5)
addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
]
loss = module(addresses).sum()
loss.backward()
print("grad_depth =", module.depth_weight.grad.item())
print("grad_parent =", module.parent_weight.grad.item())
print("grad_sibling =", module.sibling_weight.grad.item())
print("weights =", [module.depth_weight.item(), module.parent_weight.item(), module.sibling_weight.item()])


In [ ]:
# Section 5: correctness tests
import torch
from hise.address import StructuralAddress
from hise.injection import sgam_bias

# 1) Zero structural address -> zero bias -> attention identical to base model
zero_case = [StructuralAddress(0, 0, 0, 0), StructuralAddress(0, 0, 0, 0)]
zero_bias = sgam_bias(zero_case)
assert torch.allclose(zero_bias, torch.zeros_like(zero_bias))

# 2) Same d, p, s => zero penalty
addresses = [StructuralAddress(0, 1, 10, 0), StructuralAddress(2, 1, 10, 0), StructuralAddress(3, 1, 10, 0)]
assert sgam_bias(addresses)[0, 1].item() == 0.0
assert sgam_bias(addresses)[0, 2].item() == 0.0

# 3) Symmetry
mat = sgam_bias([StructuralAddress(0, 1, 10, 0), StructuralAddress(1, 1, 10, 1), StructuralAddress(2, 2, 20, 0)])
assert torch.allclose(mat, mat.T)

# 4) Boundedness for long documents
long_addresses = [StructuralAddress(i, min(20, i % 10), i * 7 % 101, i % 7) for i in range(50)]
long_bias = sgam_bias(long_addresses)
assert torch.isfinite(long_bias).all()
assert long_bias.max().item() <= 0.0
print("all SGAM invariants passed")


In [ ]:
# Section 6: SGAM matrix visualization in text form
import torch
from hise.address import StructuralAddress
from hise.injection import sgam_bias

addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
    StructuralAddress(3, 2, 21, 1),
]
mat = sgam_bias(addresses, depth_weight=-1.0, parent_weight=-2.0, sibling_weight=-0.5)
print("SGAM matrix (text heatmap):")
print(mat.detach())
print("same-section pairs are discounted less than cross-section pairs")


## 7. Implementation notes for bge-base-en-v1.5

In the real model, the SGAM layer should be inserted inside each `BertSelfAttention.forward()` immediately before the softmax. The hook pattern is:

``scores = self.matmul_qk / math.sqrt(self.head_dim)
sgam = self.sgam_bias(token_addresses)
attn = softmax(scores + sgam, dim=-1)
attn = attn.matmul(self.value)``

Recommended default: shared SGAM weights across all 12 layers for the first pass, with per-layer SGAM as an ablation and extension.

This yields a clean separation between the base BGE encoder and the document-structural prior.


## 8. Phase 2 execution plan

1. Reproduce the SGAM invariants in this notebook.
2. Validate gradient flow on the learnable weights.
3. Move to the full BGE attention hook implementation in the training notebook.
4. Build the StructEval benchmark.
5. Run shared-vs-per-layer SGAM ablation before the final fine-tuning run.
